In [ ]:
"""
Preprocessing Pipeline — Step 0

Cloud masking of Landsat 9 TIR data using QA_PIXEL band only.
 
WHY THIS STEP EXISTS:
    Your Landsat 9 scene has 17% cloud cover. Cloudy pixels contain
    no real surface temperature information — they represent cloud top
    temperatures, not the river surface. If we feed these into the SR
    model, it learns to super-resolve clouds instead of thermal structure.
    We must remove them before any further processing.
 
HOW IT WORKS:
    Landsat Collection 2 provides a QA_PIXEL band alongside every scene.
    This is a 16-bit integer band where each bit position flags a specific
    condition (cloud, shadow, snow, etc.) for every pixel. We extract the
    relevant bits and build a mask of invalid pixels.
 
Inputs:
    LC09_..._ST_B10.TIF    — Surface Temperature band (your actual LR thermal data)
    LC09_..._QA_PIXEL.TIF  — Pixel quality flags
 
Output:
    L9_TIR_masked.tif      — ST_B10 converted to Kelvin, clouds set to NoData
                             Ready to feed into Step 1 (alignment pipeline)

"""

In [1]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from datetime import datetime

In [2]:
ST_B10_PATH = "/home/ogallo/Documents/CDE/MSC_thesis/Superresolution-TIR/TIR+LS_test/LC08_L2SP_197028_20230717_20230725_02_T1/LC08_L2SP_197028_20230717_20230725_02_T1_ST_B10.TIF"      # Surface temperature band
QA_PATH     = "/home/ogallo/Documents/CDE/MSC_thesis/Superresolution-TIR/TIR+LS_test/LC08_L2SP_197028_20230717_20230725_02_T1/LC08_L2SP_197028_20230717_20230725_02_T1_QA_PIXEL.TIF"    # Pixel quality band
OUTPUT_PATH = "/home/ogallo/Documents/CDE/MSC_thesis/Superresolution-TIR/TIR+LS_test/processed/L8_TIR_masked.tif"   # Output masked TIR

In [3]:
# CELL 3 — Read all constants from MTL file
# No hardcoded scale factors or thresholds.
# Everything is retrieved from the MTL metadata file that
# comes with every Landsat download, so this code works
# automatically on any Landsat Collection 2 scene.
 
def parse_mtl(mtl_path):
    """
    Reads a Landsat _MTL.txt file and returns a dictionary of
    key-value pairs. Values are cast to float where possible,
    otherwise kept as strings (e.g. dates, scene IDs).
    """
    mtl = {}
    with open(mtl_path, "r") as f:
        for line in f:
            line = line.strip()
            # Skip section headers, group markers, and empty lines
            if "=" not in line or line.startswith("GROUP") or line.startswith("END"):
                continue
            key, value = line.split("=", 1)
            key   = key.strip()
            value = value.strip().strip('"')
            try:
                mtl[key] = float(value)
            except ValueError:
                mtl[key] = value
    return mtl
 
 
# Locate MTL file automatically in the same folder as ST_B10
# Landsat always bundles the MTL file in the same scene folder
scene_folder = os.path.dirname(ST_B10_PATH)
mtl_files    = [f for f in os.listdir(scene_folder) if f.endswith("_MTL.txt")]
 
if not mtl_files:
    raise FileNotFoundError(
        f"No MTL file found in {scene_folder}.\n"
        "Make sure your Landsat scene folder is complete and contains _MTL.txt."
    )
 
mtl_path = os.path.join(scene_folder, mtl_files[0])
print(f"MTL file found : {mtl_files[0]}")
 
# Parse the MTL into a dictionary
mtl = parse_mtl(mtl_path)
 
# ── ST_B10 scale factors ──────────────────────────────────
# Landsat Collection 2 stores ST_B10 as a scaled integer.
# The MTL file contains the exact multiplier and offset for
# this specific scene to convert DN → real Kelvin temperatures.
# Formula: temperature_kelvin = DN * SCALE + OFFSET
ST_B10_SCALE  = mtl["TEMPERATURE_MULT_BAND_ST_B10"]
ST_B10_OFFSET = mtl["TEMPERATURE_ADD_BAND_ST_B10"]
 
print(f"ST_B10 scale   : {ST_B10_SCALE}")
print(f"ST_B10 offset  : {ST_B10_OFFSET}")
 
# ── Physical temperature floor ────────────────────────────
# ST_B10 scene edges often contain very low DN values (1, 2, 3...)
# that are not flagged as nodata=0 but convert to physically
# impossible temperatures (~150K). We remove them using a
# season-aware physical floor derived from the acquisition date:
#   Summer (Apr-Sep) : 260K (-13°C) — nothing colder possible in summer
#   Winter (Oct-Mar) : 240K (-33°C) — allows for cold winter land surfaces
# This avoids hardcoding a single value and works across seasons.
date_str         = mtl["DATE_ACQUIRED"]
acquisition_date = datetime.strptime(date_str, "%Y-%m-%d")
month            = acquisition_date.month
 
if 4 <= month <= 9:
    PHYSICAL_MIN_K = 260.0
    season         = "Summer"
else:
    PHYSICAL_MIN_K = 240.0
    season         = "Winter"
 
print(f"Acquisition    : {date_str}  ({season})")
print(f"Physical floor : {PHYSICAL_MIN_K} K  ({PHYSICAL_MIN_K - 273.15:.1f} °C)")
 
# ── NoData output value ───────────────────────────────────
# -9999 is the universal geospatial NoData convention.
# Not scene-specific so kept as a constant here.
NODATA_VALUE = -9999.0
 
# ── QA_PIXEL bit flag positions ───────────────────────────
# These are fixed by the Landsat Collection 2 specification
# and do not change between scenes — no need to read from MTL.
# Each bit in the 16-bit QA_PIXEL band flags one condition.
BIT_FILL          = 0   # outside scene extent — no data at all
BIT_DILATED_CLOUD = 1   # buffer zone around cloud edges — still uncertain
BIT_CIRRUS        = 2   # thin high-altitude cloud
BIT_CLOUD         = 3   # definite cloud
BIT_CLOUD_SHADOW  = 4   # shadow cast by cloud onto the surface below
BIT_SNOW          = 5   # snow or ice covered pixel
# Bit 7 = water — intentionally NOT masked (the Rhône is water)
 

MTL file found : LC08_L2SP_197028_20230717_20230725_02_T1_MTL.txt
ST_B10 scale   : 0.00341802
ST_B10 offset  : 149.0
Acquisition    : 2023-07-17  (Summer)
Physical floor : 260.0 K  (-13.1 °C)


In [4]:
# QA_PIXEL encodes multiple conditions in one integer using bits.
# This function isolates a single bit from the entire array at once.
 
def extract_bit(qa_array, bit_position):
    """
    Returns a boolean array — True where the given bit is set (=1).
 
    How it works:
        Right-shift the array by bit_position so the target bit
        moves to position 0, then AND with 1 to isolate it.
 
    Example:
        QA = 0b00001000 → bit 3 is set → this pixel is a cloud
        extract_bit(qa, 3) → True
        extract_bit(qa, 4) → False
    """
    return (qa_array >> bit_position) & 1 == 1
 

In [5]:
# Read QA_PIXEL and build invalid mask
 
with rasterio.open(QA_PATH) as src:
    # QA_PIXEL is a 16-bit integer band — must read as uint16
    qa = src.read(1).astype(np.uint16)
 
# Extract each bad condition as its own boolean array
is_fill          = extract_bit(qa, BIT_FILL)
is_dilated_cloud = extract_bit(qa, BIT_DILATED_CLOUD)
is_cirrus        = extract_bit(qa, BIT_CIRRUS)
is_cloud         = extract_bit(qa, BIT_CLOUD)
is_cloud_shadow  = extract_bit(qa, BIT_CLOUD_SHADOW)
is_snow          = extract_bit(qa, BIT_SNOW)
 
# Combine all bad conditions — pixel is invalid if ANY is True
invalid_mask = (
    is_fill          |
    is_dilated_cloud |
    is_cirrus        |
    is_cloud         |
    is_cloud_shadow  |
    is_snow
)
 
# Print stats to see how much of the scene is being flagged
total   = qa.size
invalid = int(invalid_mask.sum())
valid   = total - invalid
 
print(f"Total pixels    : {total:,}")
print(f"Invalid pixels  : {invalid:,}  ({100*invalid/total:.1f}%)")
print(f"Valid pixels    : {valid:,}  ({100*valid/total:.1f}%)")
print(f"\nBreakdown per flag:")
print(f"  Fill          : {int(is_fill.sum()):,}")
print(f"  Dilated cloud : {int(is_dilated_cloud.sum()):,}")
print(f"  Cirrus        : {int(is_cirrus.sum()):,}")
print(f"  Cloud         : {int(is_cloud.sum()):,}")
print(f"  Cloud shadow  : {int(is_cloud_shadow.sum()):,}")
print(f"  Snow/ice      : {int(is_snow.sum()):,}")

Total pixels    : 60,774,591
Invalid pixels  : 27,346,546  (45.0%)
Valid pixels    : 33,428,045  (55.0%)

Breakdown per flag:
  Fill          : 20,150,443
  Dilated cloud : 1,331,174
  Cirrus        : 1,853,191
  Cloud         : 4,331,737
  Cloud shadow  : 2,167,773
  Snow/ice      : 67


In [6]:
# CELL 6 — Read ST_B10, convert to Kelvin, apply masks

with rasterio.open(ST_B10_PATH) as src:
    st_b10_raw    = src.read(1).astype(np.float32)
    meta          = src.meta.copy()
    native_nodata = src.nodata

# Mask very low DNs — these are scene edge artifacts, not real data
# DN=0 is official nodata, DN=1-10 convert to ~149K which is impossible
# We use DN <= 10 as a safe cutoff — no real thermal signal exists here
st_b10_nodata_mask = st_b10_raw <= 10

# Convert to Kelvin using scale factors from MTL
st_kelvin = st_b10_raw * ST_B10_SCALE + ST_B10_OFFSET

# Combine QA cloud mask + low DN artifact mask
# No physical temperature floor — avoids masking cold water pixels
combined_invalid = invalid_mask | st_b10_nodata_mask

# Apply NoData
st_kelvin[combined_invalid] = NODATA_VALUE

# Stats
valid_temps = st_kelvin[~combined_invalid]
valid_pct   = 100 * (~combined_invalid).sum() / st_kelvin.size
print(f"Valid pixels remaining : {valid_pct:.1f}%")
print(f"\nTemperature stats on valid pixels:")
print(f"  Min  : {valid_temps.min():.2f} K  ({valid_temps.min()-273.15:.2f} °C)")
print(f"  Max  : {valid_temps.max():.2f} K  ({valid_temps.max()-273.15:.2f} °C)")
print(f"  Mean : {valid_temps.mean():.2f} K  ({valid_temps.mean()-273.15:.2f} °C)")


Valid pixels remaining : 55.0%

Temperature stats on valid pixels:
  Min  : 241.45 K  (-31.70 °C)
  Max  : 333.98 K  (60.83 °C)
  Mean : 304.49 K  (31.34 °C)


In [7]:
# Save masked output

# Update metadata to reflect float32 output and our nodata value
# All spatial info (CRS, transform, bounds) is preserved from Cell 6
meta.update({
    "dtype"  : "float32",   # float32 because we now have real Kelvin values
    "nodata" : NODATA_VALUE,
    "count"  : 1
})
 
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with rasterio.open(OUTPUT_PATH, "w", **meta) as dst:
    dst.write(st_kelvin, 1)
 
print(f"✓ Saved : {OUTPUT_PATH}")

✓ Saved : /home/ogallo/Documents/CDE/MSC_thesis/Superresolution-TIR/TIR+LS_test/processed/L8_TIR_masked.tif


In [10]:
import numpy as np
import rasterio

with rasterio.open("/home/ogallo/Documents/CDE/MSC_thesis/Superresolution-TIR/TIR+LS_test/PDR-2023_TEMP_0.40_v4.tif") as src:
    hr = src.read(1)
    nodata = src.nodata
    
    # Check how many pixels have this nodata value
    nodata_pixels = (hr >= 3.4e+38).sum()
    valid_pixels  = (hr < 3.4e+38).sum()
    
    print(f"Nodata value    : {nodata}")
    print(f"Nodata pixels   : {nodata_pixels:,}")
    print(f"Valid pixels    : {valid_pixels:,}")
    print(f"Valid data range: {hr[hr < 3.4e+38].min():.2f} to {hr[hr < 3.4e+38].max():.2f}")

Nodata value    : 3.4028234663852886e+38
Nodata pixels   : 2,753,655,925
Valid pixels    : 372,836,192
Valid data range: 3.04 to 125.55


In [11]:
import numpy as np
import rasterio

with rasterio.open("/home/ogallo/Documents/CDE/MSC_thesis/Superresolution-TIR/TIR+LS_test/PDR-2023_TEMP_0.40_v4.tif") as src:
    hr      = src.read(1)
    valid   = hr[hr < 3.4e+38]

# Look at the distribution
print(f"Min    : {valid.min():.4f}")
print(f"Max    : {valid.max():.4f}")
print(f"Mean   : {valid.mean():.4f}")
print(f"Median : {np.median(valid):.4f}")
print(f"Std    : {valid.std():.4f}")

# Check if it could be Celsius
# Rhône in July should be ~15-25°C
print(f"\nIf Celsius — range is {valid.min():.1f}°C to {valid.max():.1f}°C")
print(f"If Kelvin  — range is {valid.min()-273.15:.1f}°C to {valid.max()-273.15:.1f}°C")

Min    : 3.0375
Max    : 125.5538
Mean   : 30.0329
Median : 28.4687
Std    : 4.8036

If Celsius — range is 3.0°C to 125.6°C
If Kelvin  — range is -270.1°C to -147.6°C
